# MANTA Data Exploration
This notebook demonstrates end-to-end data access, preprocessing, frequency decomposition, and phase folding for one Kepler target.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

from manta.data.downloader import download_kepler_tce_catalog, download_lightcurve
from manta.data.preprocessor import PreprocessingPipeline
from manta.data.frequency_decomposer import FrequencyDecomposer

## Step 1: Load DR25 catalog and choose one target-quarter pair
We sample a row with a valid Kepler ID and quarter to inspect one full preprocessing path.

In [ ]:
cache_dir = Path('data/cache')
cache_dir.mkdir(parents=True, exist_ok=True)

tce_df = download_kepler_tce_catalog(cache_dir=cache_dir)
sample = tce_df[['kepid', 'quarter']].dropna().drop_duplicates().iloc[0]
kepler_id = int(sample['kepid'])
quarter = int(sample['quarter'])
kepler_id, quarter

## Step 2: Download and visualize raw light curve
We retrieve the requested quarter using lightkurve and inspect raw flux before cleaning.

In [ ]:
lc = download_lightcurve(kepler_id=kepler_id, quarter=quarter, cache_dir=cache_dir / 'raw')
time = np.asarray(getattr(getattr(lc, 'time', None), 'value', lc.time))
flux = np.asarray(getattr(getattr(lc, 'flux', None), 'value', lc.flux))

plt.figure(figsize=(11, 3))
plt.plot(time, flux, lw=0.6)
plt.xlabel('Time [days]')
plt.ylabel('Flux')
plt.title(f'Raw Light Curve | KIC {kepler_id} Q{quarter}')
plt.grid(alpha=0.2)
plt.show()

## Step 3: Run full preprocessing pipeline
The pipeline handles NaNs, detrending/normalization, outlier clipping, and phase-folded global/local views.

In [ ]:
pipeline = PreprocessingPipeline(nan_strategy='hybrid', normalization_method='spline')
pp = pipeline.fit_transform({'time': time, 'flux': flux})

fig, axs = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axs[0].plot(pp['clean_time'], pp['clean_flux'], lw=0.7)
axs[0].set_ylabel('Clean Flux')
axs[0].grid(alpha=0.2)
axs[1].plot(pp['final_time'], pp['final_flux'], lw=0.7, color='tab:orange')
axs[1].set_xlabel('Time [days]')
axs[1].set_ylabel('Final Flux')
axs[1].grid(alpha=0.2)
plt.tight_layout()
plt.show()

## Step 4: Frequency decomposition
We decompose the cleaned sequence into granulation, asteroseismology, and starspot bands and inspect reconstruction quality.

In [ ]:
decomposer = FrequencyDecomposer(diagnostics_dir='outputs/diagnostics')
bands = decomposer.decompose(
    flux=pp['final_flux'],
    time=pp['final_time'],
    cadence_days=np.median(np.diff(pp['final_time']))
)
reconstructed = decomposer.reconstruct(bands)
max_err = np.max(np.abs(reconstructed - pp['final_flux']))
print('Reconstruction max abs error:', max_err)
decomposer.plot_decomposition(pp['final_flux'], pp['final_time'], bands, filename_stem='nb01_decomposition')

## Step 5: Inspect phase-folded global and local views
These arrays are the direct model inputs used by AstroNet and MANTA.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 3.5))
axs[0].plot(pp['global_view'], lw=0.8)
axs[0].set_title('Global View')
axs[0].set_xlabel('Bin')
axs[0].set_ylabel('Normalized Flux')
axs[0].grid(alpha=0.2)

axs[1].plot(pp['local_view'], lw=0.8, color='tab:red')
axs[1].set_title('Local View (Transit-Centered)')
axs[1].set_xlabel('Bin')
axs[1].set_ylabel('Normalized Flux')
axs[1].grid(alpha=0.2)
plt.tight_layout()
plt.show()